In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, RectBivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter
import niceplots.utils as nicepl


nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

In [ ]:
def get_specs(z, nu):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.05143

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

# Study the Response Functions for Different Lines

In [ ]:
astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
}
nu = 1.897 * u.THz
surveyspecs_CII = get_specs(np.array([1]), nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

In [ ]:
pobs_CII = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    astrodict_CII,
    surveyspecs_CII,
    pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1},
    output=["Power spectrum"],
)["Power spectrum"]
ssc_CII = scov.SuperSampleCovariance(pobs_CII)

In [ ]:
color = iter(Cs)

LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
plt.semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
plt.semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
plt.semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
plt.semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
plt.title("[CII], $z=1$")

In [ ]:
astrodict_CO21={
    "model_type": "ML",
    "model_name": "TonyLi",
    "model_par": {
        "alpha": 1.11,
        "beta": 0.6,
        "dMF": 1 * u.Msun * u.yr**-1 * u.Lsun**-1,
        "sig_SFR":0,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
} # CO(2-1)

nu = 2 * 115.27 * u.GHz # CO(2-1)
surveyspecs_CO21 = get_specs(np.array([1]),nu)

In [ ]:
pobs_CO21 = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    astrodict_CO21,
    surveyspecs_CO21,
    pobs_settings={"mu_kind":"gauss", "kmin":5e-3 * u.Mpc**-1},
    output=["Power spectrum"],
)["Power spectrum"]
ssc_CO21 = scov.SuperSampleCovariance(pobs_CO21)

In [ ]:
color = iter(Cs)

LiG = ssc_CO21.linear_growth_response    (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
BiC = ssc_CO21.biased_clustering_response(pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
HSV = ssc_CO21.halo_sample_variance      (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
plt.semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
plt.semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
plt.semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
plt.semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
plt.legend()
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
plt.title("CO(2-1), $z=1$")

In [ ]:
fw, fh=plt.rcParams['figure.figsize']
fig, axs=plt.subplots(1,2, figsize=(2*fw, 1.1*fh), sharex=True, sharey=True) 

color = iter(Cs)
LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
axs[0].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[0].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[0].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[0].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[0].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[0].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[0].set_title("[CII], $z=1$")

color = iter(Cs)
LiG = ssc_CO21.linear_growth_response    (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
BiC = ssc_CO21.biased_clustering_response(pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
HSV = ssc_CO21.halo_sample_variance      (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
axs[1].semilogx(pobs_CII.k, LiG, c=next(color), label="Beat Coupling")
axs[1].semilogx(pobs_CII.k, BiC, c=next(color), label="Biased Clustering")
axs[1].semilogx(pobs_CII.k, HSV, c=next(color), label="Halo Sample Variance")
axs[1].semilogx(pobs_CII.k, LiG + BiC + HSV, c=next(color), label="Total")
axs[1].set_xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
axs[1].set_ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
axs[1].set_title("CO(2-1), $z=1$")

handles, labels = [], []

for ax in [axs[0]]:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

fig.legend(handles, labels, loc='upper center', ncol=4)
plt.tight_layout(rect=[0, 0, 1, 0.93])

In [ ]:
LiG = ssc_CII.linear_growth_response    (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
BiC = ssc_CII.biased_clustering_response(pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
HSV = ssc_CII.halo_sample_variance      (pobs_CII.k, pobs_CII.z) / pobs_CII.Pk_0bs
total_CII = LiG + BiC + HSV

LiG = ssc_CO21.linear_growth_response    (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
BiC = ssc_CO21.biased_clustering_response(pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
HSV = ssc_CO21.halo_sample_variance      (pobs_CO21.k, pobs_CO21.z) / pobs_CO21.Pk_0bs
total_CO21 = LiG + BiC + HSV

color = iter(Cp)
plt.semilogx(pobs_CII.k, total_CII, label="CII", c=next(color))
plt.semilogx(pobs_CII.k, total_CO21, label="CO(2-1)", c=next(color), ls="--")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}/\mathrm{dlog}\delta_\mathrm{b}$")
plt.legend()
plt.title("$z=1$")